In [ ]:
# @title Mise en place de l'environnement
!pip install -q "evidently>=0.7,<0.8"
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh ames
import random
import warnings

import nltk
import pandas
import sklearn.feature_extraction.text
import sklearn.linear_model
import sklearn.model_selection
import sklearn.pipeline

warnings.filterwarnings("ignore")

nltk.download("words")
nltk.download("wordnet")
nltk.download("omw-1.4")


def preprocess(train_file, test_file):
  train_X = pandas.read_csv(train_file, index_col="Id")
  test_X = pandas.read_csv(test_file, index_col="Id")

  train_y = train_X.pop("SalePrice")

  all_X = pandas.concat([train_X, test_X])

  cols_1 = ["LotFrontage"]
  all_X[cols_1] = all_X[cols_1].fillna(train_X[cols_1].median())

  cols_2 = [
    "MSZoning",
    "Electrical",
    "KitchenQual",
    "Exterior1st",
    "Exterior2nd",
    "SaleType",
    "Utilities",
  ]
  all_X[cols_2] = all_X[cols_2].fillna(train_X[cols_2].mode().iloc[0, :])

  cols_4 = [
    "GarageYrBlt",
    "GarageArea",
    "GarageCars",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtFullBath",
    "BsmtHalfBath",
    "BsmtUnfSF",
    "MasVnrArea",
    "TotalBsmtSF",
  ]
  all_X[cols_4] = all_X[cols_4].fillna(0)

  cols_5 = ["Functional"]
  all_X[cols_5] = all_X[cols_5].fillna("Typ")

  all_X = all_X.fillna("NA")

  cols_numerical2label = ["MSSubClass"]
  all_X[cols_numerical2label] = all_X[cols_numerical2label].astype(str)

  quality_mapping = dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5)
  quality_columns = [
    "BsmtCond",
    "BsmtQual",
    "ExterCond",
    "ExterQual",
    "FireplaceQu",
    "GarageCond",
    "GarageQual",
    "HeatingQC",
    "KitchenQual",
    "PoolQC",
  ]
  street_mapping = dict(NA=0, Grvl=1, Pave=2)
  bsmt_fin_mapping = dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6)

  replace_mapping = dict(
    Alley=street_mapping,
    BsmtExposure=dict(NA=0, No=1, Mn=2, Av=3, Gd=4),
    BsmtFinType1=bsmt_fin_mapping,
    BsmtFinType2=bsmt_fin_mapping,
    Functional=dict(Sal=1, Sev=2, Maj2=3, Maj1=4, Mod=5, Min2=6, Min1=7, Typ=8),
    LandSlope=dict(Sev=1, Mod=2, Gtl=3),
    LotShape=dict(IR3=1, IR2=2, IR1=3, Reg=4),
    PavedDrive=dict(NA=0, N=1, P=2, Y=3),
    Street=dict(Grvl=1, Pave=2),
    Utilities=dict(ELO=1, NoSeWa=2, NoSewr=3, AllPub=4),
  )

  for quality_column in quality_columns:
    replace_mapping[quality_column] = quality_mapping

  all_X.replace(replace_mapping, inplace=True)

  print(f"Nombre de valeurs manquantes : {all_X.isnull().sum().sum()}")

  return (
    all_X.iloc[: train_X.shape[0], :],
    train_y,
    all_X.iloc[train_X.shape[0] :, :],
  )


def download_medicine_reviews() -> tuple[pandas.DataFrame, pandas.DataFrame]:
  """Data source: https://archive.ics.uci.edu/ml/datasets/Drug+Review+Dataset+%28Drugs.com%29

  Citation:
    Felix Gräßer, Surya Kallumadi, Hagen Malberg, and Sebastian Zaunseder.
    2018.
    Aspect-Based Sentiment Analysis of Drug Reviews Applying Cross-Domain and Cross-Data Learning.
    In Proceedings of the 2018 International Conference on Digital Health (DH '18).
    ACM, New York, NY, USA, 121-125.
  """
  raw_data = pandas.read_csv(
    "https://github.com/shuuchuu/datasets/raw/refs/heads/main/drug-reviews/drugsComTest_raw.tsv",
    sep="\t",
    index_col=0,
  )
  return raw_data[["drugName", "condition", "review", "rating"]]


def filter_medicine_reviews(df: pandas.DataFrame, condition: str) -> pandas.DataFrame:
  df = df.loc[
    (df["condition"] == condition) & (df["rating"].isin([1, 10])),
    ["review", "rating"],
  ]
  df["is_positive"] = df["rating"].apply(lambda x: 0 if x == 1 else 1)
  return df.drop(columns="rating")


def split_medicine_reviews(
  df: pandas.DataFrame,
) -> tuple[pandas.DataFrame, pandas.DataFrame]:
  X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    df["review"], df["is_positive"], test_size=0.4, random_state=42, shuffle=True
  )

  reference = pandas.DataFrame({"review": X_train, "is_positive": y_train})
  valid = pandas.DataFrame({"review": X_test, "is_positive": y_test})
  return reference, valid

# Monitoring — Dérive des données et des concepts

## Prise en main de la bibliothèque `evidently`

### Chargement des données

In [ ]:
reference_df, _, current_df = preprocess("ames/train.csv", "ames/test.csv")

### Imports

In [ ]:
from evidently import (
  BinaryClassification,
  DataDefinition,
  Dataset,
  Report,
)
from evidently.descriptors import (
  NonLetterCharacterPercentage,
  OOVWordsPercentage,
  SentenceCount,
  TextLength,
  WordCount,
)
from evidently.metrics import DriftedColumnsCount, UniqueValueCount, ValueDrift
from evidently.presets import (
  ClassificationPreset,
  DataDriftPreset,
  ValueStats,
)
from evidently.tests import lt, lte

### Création d'un `Dataset` muni d'une `DataDefinition`

Créez deux `Dataset` avec `Dataset.from_pandas` : `reference` pour `reference_df` et `current` pour `current_df`. La `DataDefinition` par défaut suffit : evidently déduit le type de chaque colonne de ses valeurs.

**Documentation utile**

- https://docs.evidentlyai.com/docs/library/data_definition

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `Dataset.from_pandas(df, data_definition=DataDefinition())` : un `Dataset` par `DataFrame`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
reference = Dataset.from_pandas(..., data_definition=DataDefinition())
current = ...
```

#### Solution

In [ ]:
reference = Dataset.from_pandas(reference_df, data_definition=DataDefinition())
current = Dataset.from_pandas(current_df, data_definition=DataDefinition())

### Rapports

Evidently produit des rapports, auxquels on peut attacher des tests.

Un rapport se lit : un humain y étudie les distributions et les scores de chaque colonne. Un test est une condition vérifiée automatiquement sur une métrique du rapport, par exemple pour arrêter une étape d'un pipeline ou déclencher un réentraînement.

Commencez par un rapport de dérive des données, avec le preset `DataDriftPreset` et ses paramètres par défaut, qui compare `current` à `reference`.

**Documentation utile**

- https://docs.evidentlyai.com/docs/library/report
- https://docs.evidentlyai.com/metrics/preset_data_drift

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `Report([...])` : une liste de métriques ou de presets
- `report.run(current, reference)` : les données récentes d'abord, puis la référence ; il renvoie un snapshot, que le notebook affiche s'il est la dernière expression de la cellule

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
report = Report([...])
snapshot = report.run(..., ...)
snapshot
```

#### Solution

In [ ]:
report = Report([DataDriftPreset()])
snapshot = report.run(current, reference)
snapshot

### Étudier des colonnes en particulier

Dans le jeu de données que nous utilisons pour ces travaux pratiques (AMES), deux colonnes sont particulièrement importantes : `OverallQual` et `GrLivArea`.

Créez un rapport sur ces deux colonnes avec la métrique `UniqueValueCount` pour la colonne `OverallQual` et le preset `ValueStats` pour la colonne `GrLivArea`.

**Documentation utile**

- https://docs.evidentlyai.com/metrics/all_metrics

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `UniqueValueCount(column="OverallQual")` : le nombre de lignes de chaque valeur
- `ValueStats(column="GrLivArea")` : les statistiques d'une colonne (moyenne, quantiles, valeurs manquantes…)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
report = Report([UniqueValueCount(column=...), ValueStats(column=...)])
snapshot = report.run(current, reference)
snapshot
```

#### Solution

In [ ]:
report = Report(
  [UniqueValueCount(column="OverallQual"), ValueStats(column="GrLivArea")]
)
snapshot = report.run(current, reference)
snapshot

### Configurer et sauvegarder un rapport

Par défaut, `DataDriftPreset` choisit la méthode de chaque colonne d'après son type et le nombre de lignes, et déclare le jeu en dérive quand la moitié de ses colonnes dérivent. Configurez-le : la méthode `psi` pour toutes les colonnes, et un jeu en dérive dès que 30 % de ses colonnes dérivent (`drift_share`). Affichez le nombre et la part de colonnes en dérive. Est-ce surprenant, sachant que les jeux d'entraînement et de test d'AMES viennent des mêmes ventes ?

Sauvegardez ensuite ce rapport en JSON, pour un programme, et en HTML, pour un humain.

**Documentation utile**

- https://docs.evidentlyai.com/metrics/customize_data_drift
- https://docs.evidentlyai.com/docs/library/output_formats

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `DataDriftPreset(method="psi", drift_share=0.3)` : `psi` vaut pour les colonnes numériques et catégorielles, alors que `wasserstein` ne s'applique qu'aux numériques (`num_method="wasserstein"`, avec un `cat_method` pour les autres)
- `snapshot.dict()["metrics"][0]["value"]` : la première métrique du preset, `DriftedColumnsCount`, avec le nombre (`count`) et la part (`share`) de colonnes en dérive
- `snapshot.save_json("data-drift-report.json")`, `snapshot.save_html("data-drift-report.html")`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
report = Report([DataDriftPreset(method=..., drift_share=...)])
snapshot = report.run(current, reference)
print(snapshot.dict()["metrics"][0]["value"])
snapshot.save_json(...)
snapshot.save_html(...)
```

#### Solution

In [ ]:
report = Report([DataDriftPreset(method="psi", drift_share=0.3)])
snapshot = report.run(current, reference)
# Aucune colonne ne dérive : les deux jeux sont tirés des mêmes ventes
print(snapshot.dict()["metrics"][0]["value"])
snapshot.save_json("data-drift-report.json")
snapshot.save_html("data-drift-report.html")

### Tests

Les tests sont plus adaptés que la lecture d'un rapport à un contexte automatisé comme la CI/CD. Ils s'attachent aux métriques d'un rapport, par leur paramètre `tests` : une condition (`lt`, `lte`, `gt`, `gte`, `eq`…) sur la valeur de la métrique.

Créez un rapport dont les tests vérifient qu'au plus 5 colonnes dérivent (`DriftedColumnsCount`), et que le score de dérive de `OverallQual` et celui de `GrLivArea` (`ValueDrift`) restent sous 0,05.

**Documentation utile**

- https://docs.evidentlyai.com/docs/library/tests
- https://docs.evidentlyai.com/metrics/all_metrics#data-drift

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `DriftedColumnsCount(tests=[lte(5)])` : un test sur le nombre de colonnes en dérive (`share_tests=[...]` en teste la part)
- `ValueDrift(column="OverallQual", tests=[lt(0.05)])` : un test sur le score de dérive d'une colonne (ici une distance, que la méthode par défaut choisit d'après le type de la colonne)
- `Report([...], include_tests=True)` attacherait à la place leurs tests par défaut aux métriques qui n'en ont pas

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
report = Report(
  [
    DriftedColumnsCount(tests=[...]),
    ValueDrift(column=..., tests=[...]),
    ValueDrift(column=..., tests=[...]),
  ]
)
snapshot = report.run(current, reference)
snapshot
```

#### Solution

In [ ]:
report = Report(
  [
    DriftedColumnsCount(tests=[lte(5)]),
    ValueDrift(column="OverallQual", tests=[lt(0.05)]),
    ValueDrift(column="GrLivArea", tests=[lt(0.05)]),
  ]
)
snapshot = report.run(current, reference)
snapshot

### Analyse des résultats en Python

Un pipeline lit le résultat des tests en Python. À partir de `snapshot.dict()`, calculez le pourcentage de tests réussis du dernier rapport, et affichez le nom des tests échoués.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `snapshot.dict()["tests"]` : une liste de dictionnaires, un par test, avec les clés `name`, `description` et `status` (`"SUCCESS"` ou `"FAIL"`)
- sur le rapport précédent, un des trois tests échoue

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
tests = snapshot.dict()["tests"]
successes = sum(test["status"] == ... for test in tests)
print(...)
print([test["name"] for test in tests if ...])
```

#### Solution

In [ ]:
tests = snapshot.dict()["tests"]
successes = sum(test["status"] == "SUCCESS" for test in tests)
print(f"Tests réussis : {successes / len(tests):.0%}")
print("Tests échoués :", [test["name"] for test in tests if test["status"] == "FAIL"])

## Application de la bibliothèque `evidently` à des données textuelles

### Chargement des données

Les données que nous allons utiliser sont des avis à propos de médicaments traitant plusieurs pathologies. Nous allons pour l'instant nous intéresser aux médicaments traitant la douleur, *Pain* dans les données.

In [ ]:
raw_data = download_medicine_reviews()
filtered_data = filter_medicine_reviews(raw_data, "Pain")
reference_df, current_df = split_medicine_reviews(filtered_data)

In [ ]:
reference_df

### Entraînement d'un modèle de classification

In [ ]:
pipeline = sklearn.pipeline.Pipeline(
  [
    (
      "vectorization",
      sklearn.feature_extraction.text.TfidfVectorizer(
        sublinear_tf=True, max_df=0.5, stop_words="english"
      ),
    ),
    (
      "classification",
      sklearn.linear_model.SGDClassifier(
        alpha=0.0001,
        max_iter=50,
        penalty="l1",
        loss="modified_huber",
        random_state=42,
      ),
    ),
  ]
)
pipeline.fit(reference_df["review"].values, reference_df["is_positive"].values)

Création d'une nouvelle colonne dans `reference_df` & `current_df` qui contient les prédictions :

In [ ]:
reference_df["predictions"] = pipeline.predict(reference_df["review"].values)
current_df["predictions"] = pipeline.predict(current_df["review"].values)

In [ ]:
reference_df

Définissez deux `Dataset` (`reference` et `current`) en utilisant une `DataDefinition` adaptée aux trois colonnes `review`, `is_positive` et `predictions` de nos données : une colonne de texte, et une classification binaire (sa cible et ses prédictions).

**Documentation utile**

- https://docs.evidentlyai.com/docs/library/data_definition

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `DataDefinition(text_columns=[...], classification=[...])`
- `BinaryClassification(target="is_positive", prediction_labels="predictions")` : des prédictions sous forme d'étiquettes (0 ou 1), et non de probabilités

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
data_definition = DataDefinition(
  text_columns=[...],
  classification=[BinaryClassification(target=..., prediction_labels=...)],
)
reference = Dataset.from_pandas(reference_df, data_definition=data_definition)
current = ...
```

#### Solution

In [ ]:
data_definition = DataDefinition(
  text_columns=["review"],
  classification=[
    BinaryClassification(target="is_positive", prediction_labels="predictions")
  ],
)

reference = Dataset.from_pandas(reference_df, data_definition=data_definition)
current = Dataset.from_pandas(current_df, data_definition=data_definition)

### Rapport de qualité de classification

Créez un rapport de qualité de classification, avec le preset `ClassificationPreset`, qui compare les données de validation (`current`) aux données de référence.

**Documentation utile**

- https://docs.evidentlyai.com/metrics/preset_classification

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `Report([ClassificationPreset()])`, puis `report.run(current, reference)` comme pour la dérive

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
report = Report([...])
snapshot = report.run(..., ...)
snapshot
```

#### Solution

In [ ]:
report = Report([ClassificationPreset()])
snapshot = report.run(current, reference)
snapshot

### Détection d'une dérive « technique » des données

Nous allons simuler un événement courant dans une chaîne de traitement : un bug entraîne des traitements de mauvaise qualité. Ici nous allons même en simuler deux :

- Bug dans le nettoyage des balises HTML pendant le prétraitement des avis
- Bug de récupération des données entraînant des données dans une langue différente de la langue d'entraînement

In [ ]:
# La traduction automatique en français de chaque avis sur la douleur : la
# « langue différente » du bug de récupération des données
translated_reviews = pandas.read_csv(
  "https://github.com/shuuchuu/datasets/raw/refs/heads/main/drug-reviews/drugsComTest_pain_fr.tsv",
  sep="\t",
  index_col="id",
)["review"]


random_html_tags = (
  "<body>",
  "</body>",
  "<html><body>",
  "</body></html>",
  "<h1>",
  "</h1>",
  '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 0 0" width="0" height="0" focusable="false" role="none" style="visibility: hidden; position: absolute; left: -9999px; overflow: hidden;"><defs><filter id="wp-duotone-magenta-yellow"><feColorMatrix color-interpolation-filters="sRGB" type="matrix" values=" .299 .587 .114 0 0 .299 .587 .114 0 0 .299 .587 .114 0 0 .299 .587 .114 0 0 "></feColorMatrix><feComponentTransfer color-interpolation-filters="sRGB"><feFuncR type="table" tableValues="0.78039215686275 1"></feFuncR><feFuncG type="table" tableValues="0 0.94901960784314"></feFuncG><feFuncB type="table" tableValues="0.35294117647059 0.47058823529412"></feFuncB><feFuncA type="table" tableValues="1 1"></feFuncA></feComponentTransfer><feComposite in2="SourceGraphic" operator="in"></feComposite></filter></defs></svg>',
)


def inject_random_html_tags(s):
  num_tags = 25
  for i in range(num_tags):
    random.seed(i)
    pos = random.choice(range(len(s)))
    s = s[:pos] + random.choice(random_html_tags) + s[pos:]

  return s

In [ ]:
current_disturbed_df = current_df[["review", "is_positive"]].copy()

In [ ]:
disturbed_num = int(len(current_disturbed_df) * 0.5)
random.seed(42)
disturbed_ind = random.sample(list(current_disturbed_df.index), k=disturbed_num)
current_disturbed_df.loc[disturbed_ind[: int(disturbed_num / 10)], "review"] = (
  current_disturbed_df.loc[disturbed_ind[: int(disturbed_num / 10)], "review"].apply(
    inject_random_html_tags
  )
)
translated = disturbed_ind[int(disturbed_num / 10) :]
current_disturbed_df.loc[translated, "review"] = translated_reviews.loc[translated]

In [ ]:
current_disturbed_df["predictions"] = pipeline.predict(
  current_disturbed_df["review"].values
)
current_disturbed = Dataset.from_pandas(
  current_disturbed_df, data_definition=data_definition
)

### Production d'un nouveau rapport de qualité

Reprenez le code précédent pour analyser les performances du modèle sur ces données détériorées (`current_disturbed`), en les comparant cette fois aux données de validation « propres » (`current`).

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- le même rapport, avec `current_disturbed` comme données récentes et `current` comme référence

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
report = Report([ClassificationPreset()])
snapshot = report.run(..., ...)
snapshot
```

#### Solution

In [ ]:
report = Report([ClassificationPreset()])
snapshot = report.run(current_disturbed, current)
snapshot

### Analyse des mauvaises performances du modèle

Produisez un rapport de dérive des données limité aux colonnes `is_positive` et `predictions` et aux descripteurs textuels de la colonne `review`, qui compare `current_disturbed` à `current`.

Définissez d'abord une liste `descriptors` de descripteurs : la part de caractères qui ne sont pas des lettres, la part de mots hors vocabulaire, le nombre de phrases, la longueur du texte et le nombre de mots. Ajoutez-les à `current` et à `current_disturbed` avec leur méthode `add_descriptors`.

**Documentation utile**

- https://docs.evidentlyai.com/docs/library/descriptors
- https://docs.evidentlyai.com/metrics/all_descriptors

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `NonLetterCharacterPercentage`, `OOVWordsPercentage`, `SentenceCount`, `TextLength`, `WordCount` : chacun prend la colonne de texte, `"review"`, et un `alias`, le nom de sa colonne ; la suite du notebook utilise les alias `non_letters`, `oov`, `sentence_count`, `text_length` et `word_count`
- `DataDriftPreset(columns=[...])` : la liste des colonnes à comparer, alias des descripteurs compris

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
descriptors = [
  NonLetterCharacterPercentage("review", alias="non_letters"),
  OOVWordsPercentage("review", alias=...),
  ...,
]
current.add_descriptors(descriptors)
current_disturbed.add_descriptors(...)
report = Report([DataDriftPreset(columns=["is_positive", "predictions", ...])])
snapshot = report.run(current_disturbed, current)
snapshot
```

#### Solution

In [ ]:
descriptors = [
  NonLetterCharacterPercentage("review", alias="non_letters"),
  OOVWordsPercentage("review", alias="oov"),
  SentenceCount("review", alias="sentence_count"),
  TextLength("review", alias="text_length"),
  WordCount("review", alias="word_count"),
]
descriptor_columns = [
  "non_letters",
  "oov",
  "sentence_count",
  "text_length",
  "word_count",
]

current.add_descriptors(descriptors)
current_disturbed.add_descriptors(descriptors)

report = Report(
  [DataDriftPreset(columns=["is_positive", "predictions", *descriptor_columns])]
)
snapshot = report.run(current_disturbed, current)
snapshot

### Inspection manuelle des exemples fautifs

On peut constater l'augmentation des textes longs et de la présence de mots hors vocabulaire (*OOV*).

Utilisez l'export des descripteurs en dataframe pour observer les exemples fautifs, par exemple :

- Les avis qui ont une longueur supérieure à 1000 caractères
- Les avis qui ont plus de 30 % des mots hors du vocabulaire. Comment `evidently` définit-il le vocabulaire ?

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `current_disturbed.as_dataframe()` : les colonnes du `Dataset`, plus une par descripteur, nommée d'après son alias
- pour le vocabulaire, le code d'`OOVWordsPercentage` est dans le paquet installé (`evidently/legacy/features/OOV_words_percentage_feature.py`), et la cellule de mise en place télécharge deux corpus de nltk

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
descriptors_df = current_disturbed.as_dataframe()
descriptors_df[descriptors_df["text_length"] > ...]
```

#### Solution

In [ ]:
descriptors_df = current_disturbed.as_dataframe()
descriptors_df

In [ ]:
descriptors_df[descriptors_df["text_length"] > 1000]

In [ ]:
# Le vocabulaire d'evidently est le corpus `words` de nltk (236 736 mots anglais) :
# chaque mot, débarrassé de ce qui n'est ni lettre ni chiffre et mis en minuscules,
# est ramené à son lemme par le lemmatiseur WordNet. Un nombre, un nom de
# médicament ou un mot français est donc hors vocabulaire.
descriptors_df[descriptors_df["oov"] > 30]

### Dérive progressive des données

En production, la population change rarement d'un coup. Simulons des lots de 300 avis où la part d'avis sur des médicaments contre la dépression, *Depression* dans les données, monte de 0 à 100 % : le modèle, entraîné sur la douleur, voit arriver des avis d'un autre domaine.

In [ ]:
new_content_df = filter_medicine_reviews(raw_data, "Depression")
new_content_df["predictions"] = pipeline.predict(new_content_df["review"].values)

reference.add_descriptors(descriptors)
monitored_columns = ["review", "predictions", *descriptor_columns]


def make_batch(depression_share: float, size: int = 300) -> Dataset:
  n_depression = int(size * depression_share)
  batch_df = pandas.concat(
    [
      current_df.sample(size - n_depression, random_state=42),
      new_content_df.sample(n_depression, random_state=42),
    ]
  )
  return Dataset.from_pandas(
    batch_df, data_definition=data_definition, descriptors=descriptors
  )

### Décider d'un réentraînement

En production, les étiquettes arrivent bien après les prédictions (quand elles arrivent) : la décision de réentraîner se prend sur les entrées et les prédictions seules. Créez un rapport `drift_report` dont les tests vérifient que :

- moins de 30 % des colonnes surveillées (`monitored_columns`) dérivent ;
- le score de dérive du texte de la colonne `review` reste sous 0,6.

Écrivez ensuite `should_retrain(snapshot)`, qui renvoie `True` si l'un des tests du snapshot a échoué. Essayez-la sur `make_batch(0)` et sur `make_batch(1)`, face aux données d'entraînement (`reference`), la référence du monitoring en production.

**Documentation utile**

- https://docs.evidentlyai.com/docs/library/tests

In [ ]:
# Votre code ici


def should_retrain(snapshot) -> bool: ...

#### Conseils supplémentaires

- `DriftedColumnsCount(columns=monitored_columns, share_tests=[lt(0.3)])` : un test sur la part de colonnes en dérive, et non sur leur nombre
- `ValueDrift(column="review", tests=[lt(0.6)])` : sur une colonne de texte, evidently entraîne un classifieur à distinguer les textes récents de ceux de référence ; son score est l'AUC ROC de ce classifieur (0,5 : indiscernables)
- `snapshot.dict()["tests"]` et leur `status`, comme plus haut ; sur `make_batch(0)`, `should_retrain` renvoie `False`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
drift_report = Report(
  [
    DriftedColumnsCount(columns=..., share_tests=[...]),
    ValueDrift(column="review", tests=[...]),
  ]
)


def should_retrain(snapshot) -> bool:
  return any(test["status"] == ... for test in snapshot.dict()["tests"])


print(should_retrain(drift_report.run(make_batch(0), reference)))
print(should_retrain(drift_report.run(make_batch(1), reference)))
```

#### Solution

In [ ]:
drift_report = Report(
  [
    DriftedColumnsCount(columns=monitored_columns, share_tests=[lt(0.3)]),
    ValueDrift(column="review", tests=[lt(0.6)]),
  ]
)


def should_retrain(snapshot) -> bool:
  return any(test["status"] == "FAIL" for test in snapshot.dict()["tests"])


print(should_retrain(drift_report.run(make_batch(0), reference)))
print(should_retrain(drift_report.run(make_batch(1), reference)))

### Tableau de bord

Un tableau de bord suit ces indicateurs lot après lot. Pour chaque part d'avis *Depression* de `[0, 0.2, 0.4, 0.6, 0.8, 1]`, lancez `drift_report` sur `make_batch(share)` face à `reference`, et rangez dans un `DataFrame` `dashboard`, une ligne par lot :

- la part d'avis *Depression* ;
- la part de colonnes en dérive et le score de dérive de `review` ;
- la décision de `should_retrain` ;
- l'exactitude du modèle sur le lot, calculée avec les étiquettes, qui en production n'arriveraient que bien plus tard.

Tracez la part de colonnes en dérive, le score de `review` et l'exactitude en fonction de la part d'avis *Depression*. La dérive aurait-elle été repérée à temps pour réentraîner le modèle ?

**Documentation utile**

- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `snapshot.dict()["metrics"]` : les métriques de `drift_report` dans leur ordre ; `[0]["value"]["share"]` est la part de colonnes en dérive, `[1]["value"]` le score de `review`
- `batch.as_dataframe()` : les colonnes `predictions` et `is_positive` du lot, dont l'égalité moyenne est l'exactitude
- `dashboard.plot(x="depression_share", y=[...], marker="o")`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
rows = []
for share in [0, 0.2, 0.4, 0.6, 0.8, 1]:
  batch = make_batch(share)
  snapshot = drift_report.run(batch, reference)
  metrics = snapshot.dict()["metrics"]
  batch_df = batch.as_dataframe()
  rows.append(
    {
      "depression_share": share,
      "drifted_columns": ...,
      "review_drift": ...,
      "retrain": should_retrain(snapshot),
      "accuracy": (batch_df["predictions"] == ...).mean(),
    }
  )
dashboard = pandas.DataFrame(rows)
dashboard.plot(x="depression_share", y=[...], marker="o")
dashboard
```

#### Solution

In [ ]:
rows = []
for share in [0, 0.2, 0.4, 0.6, 0.8, 1]:
  batch = make_batch(share)
  snapshot = drift_report.run(batch, reference)
  metrics = snapshot.dict()["metrics"]
  batch_df = batch.as_dataframe()
  rows.append(
    {
      "depression_share": share,
      "drifted_columns": metrics[0]["value"]["share"],
      "review_drift": metrics[1]["value"],
      "retrain": should_retrain(snapshot),
      "accuracy": (batch_df["predictions"] == batch_df["is_positive"]).mean(),
    }
  )
dashboard = pandas.DataFrame(rows)
dashboard.plot(
  x="depression_share",
  y=["drifted_columns", "review_drift", "accuracy"],
  marker="o",
)
# Le texte des avis trahit le changement de domaine sans attendre les étiquettes :
# le réentraînement est demandé alors que l'exactitude n'a encore que peu baissé.
first = dashboard.loc[dashboard["retrain"], "depression_share"].min()
print(f"Réentraînement demandé dès {first:.0%} d'avis Depression")
dashboard

### Dérive des concepts

Dans une dérive des concepts, les entrées ne changent pas : c'est leur relation avec la cible, $P(y \mid x)$, qui change, et les mêmes avis ne veulent plus dire la même chose. Simulons-la sur les données de validation : les avis qui contiennent « work » (*works*, *worked*…) voient leur étiquette inversée, comme si le sens de ce mot s'était retourné pour les patients.

In [ ]:
concept_df = current_df.copy()
flipped = concept_df["review"].str.contains("work", case=False)
concept_df.loc[flipped, "is_positive"] = 1 - concept_df.loc[flipped, "is_positive"]
concept = Dataset.from_pandas(
  concept_df, data_definition=data_definition, descriptors=descriptors
)
print(f"Étiquettes inversées : {flipped.mean():.0%} des avis")

### Détecter une dérive des concepts

Lancez `drift_report` sur `concept` face à `reference`, comme pour les lots : que décide `should_retrain` ? Produisez ensuite un rapport de qualité de classification qui compare `concept` aux mêmes avis avant la dérive, `current`. Quel signal révèle cette dérive, et quand serait-il disponible en production ?

In [ ]:
# Votre code ici

#### Conseils supplémentaires

- `drift_report.run(concept, reference)` et `should_retrain`, comme pour les lots
- `Report([ClassificationPreset()])` sur `concept` et `current`, comme plus haut

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
print(should_retrain(drift_report.run(..., ...)))
report = Report([ClassificationPreset()])
snapshot = report.run(..., ...)
snapshot
```

#### Solution

In [ ]:
# Les entrées et les prédictions sont celles de `current`, qui suivent la même
# distribution que les données d'entraînement : les tests de dérive passent. Seule
# la qualité de la classification, mesurée une fois les étiquettes connues, révèle
# la dérive des concepts.
print(f"should_retrain : {should_retrain(drift_report.run(concept, reference))}")
report = Report([ClassificationPreset()])
snapshot = report.run(concept, current)
snapshot